In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML
import matplotlib.patches as mpatches

# --- DATA ---
labels = [
    "Ich kenne die Angebote und nutze sie bereits.",
    "Ich kenne die Angebote, nutze sie aber nicht.",
    "nein"
]
values = np.array([47.1, 47.1, 5.9])  # 31.9 = blue, 64.4 = orange, 3.7 = gray
colors = ["#467592", "#FF780C", "#CCCCCC"]  # switched colors

# --- CONFIGURATION ---
fps = 30
animate_duration_sec = 10
pause_duration_sec = 3
total_frames = int(fps * animate_duration_sec)
pause_frames = int(fps * pause_duration_sec)
total_frames_with_pause = total_frames + pause_frames

# --- FIGURE SETUP ---
fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(left=0.05, right=0.75, top=0.9, bottom=0.1)
fig.patch.set_facecolor("white")
ax.axis("equal")

# --- ANIMATION FUNCTION ---
def animate(frame):
    ax.clear()
    ax.axis("equal")
    ax.set_facecolor("white")

    # Progress (0 → 1)
    if frame >= total_frames:
        progress = 1
    else:
        progress = frame / total_frames

    # Convert progress to total painted percent
    painted = 100 * progress
    blue_pct, orange_pct, gray_pct = values

    # Determine visible amounts for each segment
    blue_visible = np.clip(painted, 0, blue_pct)
    orange_visible = np.clip(painted - blue_pct, 0, orange_pct)
    gray_visible = np.clip(painted - (blue_pct + orange_pct), 0, gray_pct)
    remainder = 100 - (blue_visible + orange_visible + gray_visible)

    # Actual values for this frame
    current_values = [blue_visible, orange_visible, gray_visible, remainder]
    current_colors = [colors[0], colors[1], colors[2], (0, 0, 0, 0)]  # transparent remainder

    # --- Draw donut pie ---
    wedges, _ = ax.pie(
        current_values,
        colors=current_colors,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2)
    )

    # --- Add text labels for visible slices ---
    for i, w in enumerate(wedges[:3]):  # only visible slices (no transparent remainder)
        val = current_values[i]
        if val <= 0.3:
            continue
        ang = (w.theta2 - w.theta1) / 2 + w.theta1
        x = np.cos(np.deg2rad(ang))
        y = np.sin(np.deg2rad(ang))

        # White text for blue/orange, black text for gray
        text_color = "black" if i == 2 else "white"

        ax.text(
            x * 0.75, y * 0.75, f"{val:.1f}%",
            ha="center", va="center", color=text_color, fontsize=10, fontweight="bold"
        )

    # --- LEGEND ---
    legend_patches = [
        mpatches.Patch(color=colors[0], label=labels[0]),
        mpatches.Patch(color=colors[1], label=labels[1]),
        mpatches.Patch(color=colors[2], label=labels[2])
    ]
    ax.legend(
        handles=legend_patches,
        loc="upper right",
        bbox_to_anchor=(1.35, 1.0),
        fontsize=10,
        frameon=False
    )

    return wedges

# --- CREATE ANIMATION ---
anim = FuncAnimation(
    fig,
    animate,
    frames=total_frames_with_pause,
    interval=1000 / fps,
    blit=False
)
plt.close(fig)

# --- DISPLAY INLINE ---
HTML(anim.to_html5_video())

# --- OPTIONAL SAVE ---
writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
anim.save("angebote_donut_with_full_legend.mp4", writer=writer, dpi=300)
print("✅ Saved: angebote_donut_with_full_legend.mp4")


✅ Saved: angebote_donut_with_full_legend.mp4


In [ ]:
from google.colab import files
files.download('angebote_donut_with_full_legend.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !ffmpeg -stream_loop 1 -i angebote_donut_with_full_legend.mp4 -c copy arbeitsumfeld.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Wedge
from IPython.display import HTML

# --- DATA (from image) ---
values = np.array([11.1, 22.2, 44.4, 22.2])
colors = np.array([
    [11,122,46],    # dark green
    [142,232,137],  # light green
    [243,242,191],  # yellow
    [255,0,0]      # red
]) / 255

# --- CONFIG ---
fps = 30
animate_duration_sec = 10
pause_duration_sec = 3
total_frames = int(fps * animate_duration_sec)
pause_frames = int(fps * pause_duration_sec)
total_frames_with_pause = total_frames + pause_frames

outer_radius = 1.35
outer_width = 0.07
gradient_steps = 700   # smoothness of gradient

# --- FIGURE ---
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")
ax.set_aspect("equal")
ax.axis("off")

# --- Gradient interpolation ---
def lerp(c1, c2, t):
    return c1 * (1 - t) + c2 * t


def draw_gradient_ring(ax, progress):
    total_angle = 360 * (progress / 100)
    step = total_angle / gradient_steps

    bounds = np.cumsum(np.insert(values, 0, 0)) / 100

    for i in range(gradient_steps):
        a0 = 90 - i * step
        a1 = 90 - (i + 1) * step
        p = i / gradient_steps

        for j in range(len(values)):
            if bounds[j] <= p <= bounds[j + 1]:
                local_t = (p - bounds[j]) / (bounds[j + 1] - bounds[j])
                c1 = colors[j]
                c2 = colors[min(j + 1, len(colors) - 1)]
                col = lerp(c1, c2, local_t)
                break

        ax.add_patch(Wedge((0,0), outer_radius, a1, a0,
                           width=outer_width,
                           facecolor=col,
                           edgecolor="none"))


def animate(frame):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")

    progress = 1 if frame >= total_frames else frame / total_frames
    painted = 100 * progress

    # inner donut logic
    visible = []
    r = painted
    for v in values:
        x = np.clip(r, 0, v)
        visible.append(x)
        r -= v

    remainder = 100 - sum(visible)

    ax.pie(visible + [remainder],
           colors=[*colors, (0,0,0,0)],
           startangle=90,
           counterclock=False,
           wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2))

    # gradient outer ring
    draw_gradient_ring(ax, painted)

    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.35, 1.35)


# --- RUN ---
anim = FuncAnimation(fig, animate,
                     frames=total_frames_with_pause,
                     interval=1000/fps,
                     blit=False)
plt.close(fig)

HTML(anim.to_html5_video())

# # --- SAVE ---
# writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt","yuv420p"])
# anim.save("zufriedenheit_gradient_ring.mp4", writer=writer, dpi=300)
# print("✅ Saved: zufriedenheit_gradient_ring.mp4")


CalledProcessError: Command '['ffmpeg', '-f', 'rawvideo', '-vcodec', 'rawvideo', '-s', '1000x600', '-pix_fmt', 'rgba', '-framerate', '29.999999999999996', '-loglevel', 'error', '-i', 'pipe:', '-vcodec', 'h264', '-pix_fmt', 'yuv420p', '-y', '/tmp/tmp1roknrsl/temp.m4v']' returned non-zero exit status 255.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML

# --- DATA (from image) ---
values = np.array([11.1, 22.2, 44.4, 22.2])

# colors as RGB (0..1)
colors = np.array([
    [11, 122, 46],    # dark green
    [142, 232, 137],  # light green
    [243, 242, 191],  # pale yellow
    [255, 0, 0]       # red
]) / 255.0

# --- CONFIG ---
fps = 30
animate_duration_sec = 10
pause_duration_sec = 3
total_frames = int(fps * animate_duration_sec)
pause_frames = int(fps * pause_duration_sec)
total_frames_with_pause = total_frames + pause_frames

# ring geometry (in data coords)
outer_radius = 1.35
outer_width = 0.07

# ring texture resolution (higher = smoother but still fast)
N = 900  # texture size (pixels per side)

# ---------- Build a single RGBA image of the gradient ring ----------
ys, xs = np.ogrid[-1:1:N*1j, -1:1:N*1j]
r = np.sqrt(xs**2 + ys**2)

# angle in degrees, 0 at +x axis, increasing CCW; we’ll map to start at 90 and clockwise
ang = (np.degrees(np.arctan2(ys, xs)) + 360) % 360  # 0..360

# Convert to "clockwise from 90°"
# at top (90° CCW) should be 0 progress; clockwise increases
theta = (90 - ang) % 360  # 0..360 clockwise starting at 12 o'clock
p = theta / 360.0         # 0..1 around ring

# segment boundaries in [0,1]
bounds = np.cumsum(np.insert(values, 0, 0)) / 100.0

# compute gradient color around the circle
rgb = np.zeros((N, N, 3), dtype=float)

def lerp(c1, c2, t):
    return c1 * (1 - t) + c2 * t

for j in range(len(values)):
    mask = (p >= bounds[j]) & (p <= bounds[j + 1])
    # local progress inside segment
    t = (p[mask] - bounds[j]) / (bounds[j + 1] - bounds[j] + 1e-12)
    c1 = colors[j]
    c2 = colors[(j + 1) % len(colors)]  # blend into next color (wrap-around)
    rgb[mask] = lerp(c1, c2, t[..., None])

# alpha mask: only show pixels inside the ring thickness
r_min = (outer_radius - outer_width) / outer_radius
r_max = 1.0
alpha_ring = ((r >= r_min) & (r <= r_max)).astype(float)

# start fully transparent; we’ll reveal by angle each frame
rgba = np.dstack([rgb, alpha_ring])

# ---------- Figure ----------
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")
ax.set_aspect("equal")
ax.axis("off")

extent = (-outer_radius, outer_radius, -outer_radius, outer_radius)

# draw ring image once (we update its alpha mask each frame)
ring_im = ax.imshow(rgba, extent=extent, origin="lower", interpolation="bilinear", zorder=0)

def animate(frame):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")

    progress = 1.0 if frame >= total_frames else frame / total_frames
    painted = 100 * progress

    # --- inner donut (same style as before) ---
    visible = []
    remaining = painted
    for v in values:
        vis = np.clip(remaining, 0, v)
        visible.append(vis)
        remaining -= v
    remainder = 100 - sum(visible)

    ax.pie(
        visible + [remainder],
        colors=[*colors, (0, 0, 0, 0)],
        startangle=90,
        counterclock=False,
        wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2)
    )

    # --- outer ring: update alpha to reveal only up to current angle ---
    # reveal theta <= painted% of 360
    reveal_theta = 360 * (painted / 100.0)
    alpha_reveal = ((theta <= reveal_theta).astype(float)) * alpha_ring

    rgba2 = rgba.copy()
    rgba2[..., 3] = alpha_reveal

    ax.imshow(rgba2, extent=extent, origin="lower", interpolation="bilinear", zorder=0)

    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.35, 1.35)

    return []

anim = FuncAnimation(
    fig,
    animate,
    frames=total_frames_with_pause,
    interval=1000 / fps,
    blit=False
)
plt.close(fig)

HTML(anim.to_html5_video())

# # --- SAVE ---
# writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
# anim.save("zufriedenheit_gradient_ring_fast.mp4", writer=writer, dpi=300)
# print("✅ Saved: zufriedenheit_gradient_ring_fast.mp4")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML

# --- DATA (from image) ---
values = np.array([11.1, 22.2, 44.4, 22.2])

# colors as RGB (0..1)
colors = np.array([
    [11, 122, 46],    # dark green
    [142, 232, 137],  # light green
    [243, 242, 191],  # pale yellow
    [255, 0, 0]       # red
]) / 255.0

# --- CONFIG ---
fps = 30
animate_duration_sec = 10
pause_duration_sec = 3
total_frames = int(fps * animate_duration_sec)
pause_frames = int(fps * pause_duration_sec)
total_frames_with_pause = total_frames + pause_frames

outer_radius = 1.35
outer_width = 0.07
N = 900  # ring texture resolution

# --- helper ---
def lerp(c1, c2, t):
    return c1 * (1 - t) + c2 * t

# ---------- Build a single RGBA image of the outer ring ----------
ys, xs = np.ogrid[-1:1:N*1j, -1:1:N*1j]
r = np.sqrt(xs**2 + ys**2)

ang = (np.degrees(np.arctan2(ys, xs)) + 360) % 360
theta = (90 - ang) % 360          # 0..360 clockwise from 12 o'clock
p = theta / 360.0                 # 0..1 around ring

# --- OUTER RING VISUAL WEIGHTS (only dark green longer; yellow shorter) ---
# --- OUTER RING VISUAL WEIGHTS (must sum to 100!) ---
delta = 12.0

outer_weights = np.array([
    11.1 + delta,  # dark green longer
    22.2,          # light green unchanged
    44.4 - delta,  # yellow reduced
    22.2           # red unchanged
])


bounds = np.cumsum(np.insert(outer_weights, 0, 0)) / 100.0

rgb = np.zeros((N, N, 3), dtype=float)

# Red segment index is 3 (last one)
RED_IDX = 3

# Small white separation only around red segment boundaries
gap_deg = 3.0
gap = gap_deg / 360.0

red_start = bounds[RED_IDX]
red_end = bounds[RED_IDX + 1]

gap_start_mask = (p >= (red_start - gap/2)) & (p <= (red_start + gap/2))
gap_end_mask = (p >= (red_end - gap/2)) | (p <= (gap/2))

gap_mask = gap_start_mask | gap_end_mask

for j in range(len(values)):
    mask = (p >= bounds[j]) & (p <= bounds[j + 1])

    if j == RED_IDX:
        # RED is solid (no gradient)
        rgb[mask] = colors[j]
    elif j == RED_IDX - 1:
        # Yellow does NOT blend into red (hard transition)
        rgb[mask] = colors[j]
    else:
        # gradients for green -> light green -> yellow
        t = (p[mask] - bounds[j]) / (bounds[j + 1] - bounds[j] + 1e-12)
        c1 = colors[j]
        c2 = colors[j + 1]
        rgb[mask] = lerp(c1, c2, t[..., None])

# alpha mask for ring thickness
r_min = (outer_radius - outer_width) / outer_radius
r_max = 1.0
alpha_ring = ((r >= r_min) & (r <= r_max)).astype(float)

# punch white gaps ONLY at red boundaries
alpha_ring = alpha_ring * (~gap_mask)

rgba = np.dstack([rgb, alpha_ring])

# ---------- Figure ----------
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")
ax.set_aspect("equal")
ax.axis("off")

extent = (-outer_radius, outer_radius, -outer_radius, outer_radius)

def animate(frame):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")

    progress = 1.0 if frame >= total_frames else frame / total_frames
    painted = 100 * progress

    # inner donut
    visible = []
    remaining = painted
    for v in values:
        vis = np.clip(remaining, 0, v)
        visible.append(vis)
        remaining -= v
    remainder = 100 - sum(visible)

    ax.pie(
        visible + [remainder],
        colors=[*colors, (0, 0, 0, 0)],
        startangle=90,
        counterclock=False,
        wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2)
    )

    # ✅ ADD: Percentage labels inside donut (only for visible parts)
    for i, v in enumerate(visible):
        if v <= 0.6:
            continue

        start = sum(visible[:i])
        mid = start + v / 2.0
        angle = 90 - (mid / 100.0) * 360.0

        x = np.cos(np.deg2rad(angle)) * 0.77   # move inward
        y = np.sin(np.deg2rad(angle)) * 0.77

        ax.text(
            x, y, f"{v:.1f}%",
            ha="center", va="center",
            fontsize=11, fontweight="normal",
            color="black",     # all black
            zorder=10
        )


    # outer ring reveal
    reveal_theta = 360 * (painted / 100.0)
    alpha_reveal = ((theta <= reveal_theta).astype(float)) * alpha_ring

    rgba2 = rgba.copy()
    rgba2[..., 3] = alpha_reveal

    ax.imshow(rgba2, extent=extent, origin="lower", interpolation="bilinear", zorder=0)

    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.35, 1.35)
    return []

anim = FuncAnimation(
    fig,
    animate,
    frames=total_frames_with_pause,
    interval=1000 / fps,
    blit=False
)
plt.close(fig)

# --- SAVE ---
writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
anim.save("zufriedenheit_gradient_ring_fast.mp4", writer=writer, dpi=300)
print("✅ Saved: zufriedenheit_gradient_ring_fast.mp4")

HTML(anim.to_html5_video())


✅ Saved: zufriedenheit_gradient_ring_fast.mp4


In [ ]:
from google.colab import files
files.download('zufriedenheit_gradient_ring_fast.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>